# Decision Theory in Binary Classification

We [recently
introduced](https://middcs.github.io/data-science-notes/chapters/25-classification.html)
classification, the task of predicting a discrete label from input
features. We focused on logistic regression for predicting binary
(“yes/no” or 1/0 labels). Logistic regression is one of several
*score-based linear models*, which produces a prediction by computing a
*score*

$$
\begin{aligned}
    s = \sum_{i = 1}^p w_i x_i
\end{aligned}
$$

and then predicting a label of 1 if $s$ exceeds some threshold $t$ and 0
otherwise. In this activity, we’ll explore the impact of varying the
threshold $t$ on the performance of a score-based binary classifier like
logistic regression. We’ll first consider this question from the
standpoint of error rates, which will lead us to the concept of the
*Receiver Operating Characteristic (ROC) curve*. Then, we’ll consider it
from the standpoint of decision-theory, in which the cost of being wrong
is not the same for all types of errors.

## Model Training

First, download the Australia weather data set, prepare it for binary
classification, split it into training and test sets, and train a
logistic regression model to predict whether it will rain tomorrow. It’s
fine to reuse code from lecture for this.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import numpy as np
import seaborn as sns
sns.set(style="whitegrid")

url = "https://raw.githubusercontent.com/middcs/data-science-notes/refs/heads/main/data/australia-weather/weatherAUS.csv"
df = pd.read_csv(url)
df = df.dropna()
df = df.drop(columns=["Location", "Date"])
df["RainToday"]    = df["RainToday"].map({"No": 0, "Yes": 1})
df["RainTomorrow"] = df["RainTomorrow"].map({"No": 0, "Yes": 1})

y = df["RainTomorrow"]
X = df.drop(columns=["RainTomorrow"])
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2026)

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

model = LogisticRegression()
f = model.fit(X_train, y_train)

We can extract the scores for the test set using the `predict_proba`
method of the trained model. This returns a 2-column array, where the
first column gives the predicted probability of label 0 and the second
column gives the predicted probability of label 1. We’ll extract just
the second column.

In [ ]:
y_scores = f.predict_proba(X_test)[:, 1]
y_scores

*These scores have been normalized to lie between 0 and 1.*

Recall that our decision rule is to predict label 1 if the score exceeds
some threshold $t$ and label 0 otherwise. Write a function
`predict_with_threshold` that takes as input an array of scores and a
threshold value, and returns a binary array of predictions.

In [ ]:
# TODO: Your code here

You can check your implementation by comparing the predictions from your
function with those from the model when using a threshold of 0.5 – if
the cell below returns `np.True_` then you have passed this check and
your implementation is likely correct.

In [ ]:
# model.score(X_test, y_test)
np.all(predict_with_threshold(y_scores, 0.5) == model.predict(X_test))

Now write a function called `positive_rates` which accepts a
`np.ndarray` of thresholds, the score vector, and the vector of true
labels, and returns two arrays: one containing the the true positive
rate and one containing the false positive rate for each threshold. So,
if you pass the array

``` python
t = np.linspace(0, 1, 5)
```

to

``` python
tp_rates, fp_rates = positive_rates(t, y_scores, y_test)\;,
```

you should obtain arrays `tp_rates` and `fp_rates`, each of length 5,
where `tp_rates[i]` is the true positive rate and `fp_rates[i]` is the
false positive rate when using threshold `t[i]`.

***Hint***: You are likely to require a for-loop over the thresholds.
You can use your `predict_with_threshold` function to get the predicted
labels for each threshold. Then, you can compute true positives, false
negatives, false positives, and true negatives by comparing the
predicted labels to the true labels. Finally, you can compute the true
positive rate and false positive rate using their definitions:

In [ ]:
# TODO: Your code here

### ROC Curve

Once you’ve implemented `positive_rates`, use it to compute the true
positive rates and false positive rates for thresholds ranging from 0 to
1 in increments of 0.01. Then, make a lineplot in which the horizontal
axis is the false positive rate and the vertical axis is the true
positive rate.

In [ ]:
t = np.linspace(0, 1, 101)
tp_rates, fp_rates = positive_rates(t, y_scores, y_test)

In [ ]:
# TODO: Your code here

This plot is called the Receiver Operating Characteristic (ROC) curve
for our binary classifier on this data set. The ROC curve summarizes the
performance of our classifier across possible thresholds, and indicates
the tradeoff between true positive rate and false positive rate. For
example, the ROC curve says that if we are willing to accept a false
positive rate of about `python fp_rates[50]` then we can achieve a true
positive rate of about `python tp_rates[50]`. The closer the ROC curve
comes to the top-left corner of the plot, the stronger the overall
performance of our classifier.

#### Area Under the Curve (AUC)

The Area Under The Curve (AUC) is a common measure for the overall
quality of a classifier. An ROC that achieves perfect classification
(100% true positive rate and 0% false positive rate) has an AUC of 1.0,
while a classifier that makes random predictions has an expected AUC of
0.5.

Implement a function `roc_auc` that computes the AUC for a given set of
true positive rates and false positive rates. You can use the
trapezoidal rule to approximate the area under the curve:

$$
\begin{aligned}
    \mathrm{Area} \approx \frac{1}{2}\sum_{i = 1}^{n-1} (\mathrm{TPR}(t_{i-1}) +  \mathrm{TPR}(t_{i})) \cdot \left(\mathrm{FPR}(t_{i}) - \mathrm{FPR}(t_{i-1})\right)
\end{aligned}
$$

Then, use your function to compute the AUC for the ROC curve you
generated above, recalling that an AUC close to 1.0 indicates a strong
classifier.

***Hint***: It is possible to evaluate this sum with a for-loop, but it
is also possible to use NumPy array operations to avoid explicit loops.

In [ ]:
# TODO: Your code here

## Decision Theory in Classification

In many classification applications, the costs of different types of
errors are not the same. For example, suppose you are going to use our
rain predictor to decide whether or not to bring an umbrella with you
when you leave today. There are four possible outcomes:

-   **True positive (TP)**: You predicted rain and brought your
    umbrella, and it did indeed rain! **You are dry and happy.**
-   **True negative (TN)**: You predicted no rain and did not bring your
    umbrella, and it did not rain. **You are dry and happy.**
-   **False positive (FP)**: You predicted rain and brought your
    umbrella, but it did not rain. You are dry, but you had to carry
    around an unnecessary umbrella all day and your friends made fun of
    you. **You are dry but lightly embarassed.**  
-   **False negative (FN)**: You predicted no rain and did not bring
    your umbrella, but it rained. **You are wet and very embarassed.**

When modeling these outcomes, we might reasonably think that the **TP**
and **TN** outcomes are both “good,” the **FP** outcome is “somewhat
bad,” and the **FN** outcome is “very bad.” To formalize this, we can
assign *utilities* to each outcome. For example, we might assign
utilities as follows:

| Outcome | Utility |
|---------|---------|
| ———     | ———     |
| TP      | 0       |
| TN      | 0       |
| FP      | -1      |
| FN      | -5      |

This scenario reflects *asymmetrical cost of error*: we want to avoid
false negatives much more than false positives.

Given these utilities, we can compute the *expected utility* of our
classifier at a given threshold $t$ as follows:

$$
\begin{aligned}
    \mathrm{EU}(t) = \frac{1}{n} \mathrm{FP}(t) \cdot U_{FP} + \frac{1}{n} \mathrm{FN}(t) \cdot U_{FN}\;,
\end{aligned}
$$

where $\mathrm{FP}(t)$ is the number of false positives at threshold
$t$, $\mathrm{FN}(t)$ is the number of false negatives at threshold $t$,
$U_{FP}$ is the utility of a false positive, $U_{FN}$ is the utility of
a false negative, and $n$ is the total number of predictions. We do not
include true positives or true negatives in this calculation, since
their utility is zero.

Write a function which computes the expected utility of our classifier
at each threshold in a given array of thresholds, using the utilities
that the user can pass in.

In [ ]:
# TODO: Your code here

Then, use this function to compute the expected utility at each
threshold from 0 to 1 in increments of 0.01, and make a line plot of
expected utility versus threshold. Please do this twice:

-   First, using the utilities $U_{FP} = -1$ and $U_{FN} = -5$ as in the
    table above.
-   Second, using the utilities $U_{FP} = -1$ and $U_{FN} = -2$, which
    reflects a scenario in which false negatives are still worse than
    false positives but not as dramatically so.

These two utility combinations might be used by a person who hates
getting wet (first case) and one who isn’t too bothered by getting a
little wet (second case).

In [ ]:
# TODO: Your code here

Finally, based on your plots, how does changing the relative costs of
false positives and false negatives affect the optimal threshold for
decision-making using this classifier?